<a href="https://colab.research.google.com/github/rakjarvis/AI-SMS-Spam-Filter-From-Scratch-Bayes-Theorem-/blob/main/AI_SMS_Spam_Filter_From_Scratch_Bayes_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

In [4]:
url="https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"

In [5]:
df = pd.read_csv(url, sep='\t', header=None, names=['Label', 'Text'])

In [6]:
df.head()

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
from IPython.utils import text
def clean_text(raw_text):
  lower_text=raw_text.lower()

  punctuation = "?.:,!"

  clean_text = ""
  for character in lower_text:
    if character not in punctuation:
      clean_text=clean_text+character

  return clean_text

df["clean_text"] = df["Text"].apply(clean_text)

df[["Text", "clean_text"]].head()

,Text,clean_text
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i don't think he goes to usf he lives arou...


In [8]:
df["Words"] = df["clean_text"].apply(lambda x:x.split())

df[["clean_text","Words"]].head()

,clean_text,Words
0,go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o..."
1,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]"
2,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t..."
4,nah i don't think he goes to usf he lives arou...,"[nah, i, don't, think, he, goes, to, usf, he, ..."


In [9]:
spam_pile = df[df["Label"] == "spam"]
ham_pile = df[df["Label"] == "ham"]

print("Number of Spam Emails : ", len(spam_pile))
print("Numer of Ham Emails : ", len(ham_pile))

Number of Spam Emails :  747
Numer of Ham Emails :  4825


In [10]:
spam_pile.head()

,Label,Text,clean_text,Words
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey there darling it's been 3 week's n...,"[freemsg, hey, there, darling, it's, been, 3, ..."
8,spam,WINNER!! As a valued network customer you have...,winner as a valued network customer you have b...,"[winner, as, a, valued, network, customer, you..."
9,spam,Had your mobile 11 months or more? U R entitle...,had your mobile 11 months or more u r entitled...,"[had, your, mobile, 11, months, or, more, u, r..."
11,spam,"SIX chances to win CASH! From 100 to 20,000 po...",six chances to win cash from 100 to 20000 poun...,"[six, chances, to, win, cash, from, 100, to, 2..."


In [11]:
ham_pile.head()

,Label,Text,clean_text,Words
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o..."
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]"
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t..."
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah i don't think he goes to usf he lives arou...,"[nah, i, don't, think, he, goes, to, usf, he, ..."
6,ham,Even my brother is not like to speak with me. ...,even my brother is not like to speak with me t...,"[even, my, brother, is, not, like, to, speak, ..."


In [12]:
spam_word_count = 0
for word_list in spam_pile["Words"]:
  if "free" in word_list:
    spam_word_count += 1

ham_word_count = 0
for word_list in ham_pile["Words"]:
  if "free" in word_list:
    ham_word_count += 1

prob_free_given_spam = spam_word_count / len(spam_pile)
prob_free_given_ham = ham_word_count / len(ham_pile)

print(f"probability of free in spam: {prob_free_given_spam: .2%}")
print(f"probability of free in ham: {prob_free_given_ham: .2%}")


probability of free in spam:  22.09%
probability of free in ham:  1.20%


In [13]:
spam_counts = {}
ham_counts = {}

for words_list in spam_pile["Words"]:
  for Word in words_list:
    if Word not in spam_counts:
      spam_counts[Word] = 1
    else:
      spam_counts[Word] += 1

for words_list in ham_pile["Words"]:
  for Word in words_list:
    if Word not in ham_counts:
      ham_counts[Word] = 1
    else:
      ham_counts[Word] += 1

print("Word Counting Complete")

Word Counting Complete


In [14]:
def predict_spam_or_ham(new_message):
    # 1. Clean and split the new message just like before
    clean_msg = new_message.lower().replace("!", "").replace("?", "").replace(".", "")
    words = clean_msg.split()

    # 2. Start our scores. (We start at 1.0 because we are going to multiply)
    spam_score = 1.0
    ham_score = 1.0

    # 3. Look at each word and calculate its probability
    for word in words:
        # How many times did this word appear in spam? (Default to 1 if never seen)
        word_spam_count = spam_counts.get(word, 1)
        # How many times did this word appear in ham? (Default to 1 if never seen)
        word_ham_count = ham_counts.get(word, 1)

        # Calculate the simple probability for this word
        prob_word_given_spam = word_spam_count / len(spam_pile)
        prob_word_given_ham = word_ham_count / len(ham_pile)

        # Multiply this word's probability into our total running score
        spam_score = spam_score * prob_word_given_spam
        ham_score = ham_score * prob_word_given_ham

    # 4. Compare the final scores!
    print(f"Spam Score: {spam_score}")
    print(f"Ham Score: {ham_score}")

    if spam_score > ham_score:
        return "🚨 SPAM DETECTED!"
    else:
        return "📥 This is a normal message (HAM)."

In [15]:
test_message = "URGENT! Claim your free cash prize right now!"
result = predict_spam_or_ham(test_message)
print("Result:", result)

Spam Score: 1.3365532798571213e-08
Ham Score: 5.11525804925507e-20
Result: 🚨 SPAM DETECTED!
